# Data Quality

Role: Data Engineer

Objective:
- Identify and quantify all data quality issues
- Map issues to data quality dimensions
- Produce a cleaned (curated) dataset for downstream bias analysis


In [60]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

from pathlib import Path

DATA_PATH = Path("data")
list(DATA_PATH.glob("*"))


[]

In [61]:
raw_file = data_path / "data.json"

import json
with open(raw_file, "r", encoding="utf-8") as f:
    raw = json.load(f)

len(raw), type(raw)



(502, list)

In [62]:
df = pd.json_normalize(raw, sep=".")
df.head()


,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,...,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,...,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN


In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 21 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   _id                               502 non-null    object 
 1   spending_behavior                 502 non-null    object 
 2   processing_timestamp              62 non-null     object 
 3   applicant_info.full_name          502 non-null    object 
 4   applicant_info.email              502 non-null    object 
 5   applicant_info.ssn                497 non-null    object 
 6   applicant_info.ip_address         497 non-null    object 
 7   applicant_info.gender             501 non-null    object 
 8   applicant_info.date_of_birth      501 non-null    object 
 9   applicant_info.zip_code           501 non-null    object 
 10  financials.annual_income          497 non-null    object 
 11  financials.credit_history_months  502 non-null    int64  
 12  financia

## Initial Structural Observations

The dataset contains 502 credit applications and 21 columns. 
Most fields are stored as object types, including some financial variables that should ideally be numeric.

Several immediate data quality concerns were identified:

1. **High Missingness**
   - `processing_timestamp` has very low completeness (~12% non-null).
   - `loan_purpose` is missing in the majority of records.
   - `notes` is nearly empty and likely non-essential.

2. **Schema Inconsistency**
   - Both `financials.annual_income` and `financials.annual_salary` exist.
   - These likely represent the same concept but are stored separately.
   - This creates structural inconsistency.

3. **Potential Type Issues**
   - `financials.annual_income` is stored as an object rather than numeric.
   - This suggests mixed data types (strings and numbers), which may break downstream analysis.

4. **Sensitive Personal Data**
   - The dataset contains PII such as SSN, email, IP address, and date of birth.
   - These fields will require special handling in later privacy and governance steps.


In [64]:
df.describe(include="all")

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
count,502,502,62,502,502,497,497,501,501,501,...,502.000000,502.000000,502.000000,502,210,50,292.000000,292.000000,5.000000,2
unique,500,496,4,475,494,494,496,5,494,196,...,NaN,NaN,NaN,2,4,10,NaN,NaN,NaN,2
top,app_042,"[{'category': 'Utilities', 'amount': 786}]",2024-01-15T00:00:00Z,Susan Flores,,937-72-8731,192.168.91.142,Male,,10048,...,NaN,NaN,NaN,True,algorithm_risk_score,medical,NaN,NaN,NaN,RESUBMISSION
freq,2,2,59,3,7,2,2,195,4,8,...,NaN,NaN,NaN,292,170,8,NaN,NaN,NaN,1
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,50.402390,0.246195,29493.503984,NaN,NaN,NaN,4.564726,47845.890411,69200.000000,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,31.234824,0.136296,16775.309756,NaN,NaN,NaN,1.162866,18103.754530,22664.950915,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-10.000000,0.050000,-5000.000000,NaN,NaN,NaN,2.500000,15000.000000,45000.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,27.250000,0.150000,17258.250000,NaN,NaN,NaN,3.500000,34000.000000,46000.000000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,48.000000,0.230000,27385.500000,NaN,NaN,NaN,4.550000,48000.000000,75000.000000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,72.000000,0.350000,38251.500000,NaN,NaN,NaN,5.600000,62250.000000,86000.000000,NaN


## Descriptive Overview and Early Signals

The descriptive summary confirms several important structural observations:

- The dataset contains 502 records.
- The `_id` field has 500 unique values, indicating potential duplicate application IDs.
- `processing_timestamp` appears in only 62 records, confirming severe incompleteness.
- Some categorical fields (full names, emails) contain repeated values, which may indicate duplicate entries or shared identities.

At this stage, we focus on identifying structural anomalies such as duplicate records before proceeding to type corrections and cleaning transformations.

In [65]:
df["_id"].value_counts()

_id
app_042    2
app_001    2
app_440    1
app_096    1
app_237    1
          ..
app_156    1
app_201    1
app_073    1
app_290    1
app_163    1
Name: count, Length: 500, dtype: int64

## Duplicate ID Detection

The `_id` column reveals duplicate application identifiers.

Specifically:
- `app_042` appears twice.
- `app_001` appears twice.

Since `_id` should uniquely identify each credit application, this represents a **consistency violation**.

Duplicate primary identifiers can:
- Distort approval rate calculations
- Bias fairness metrics
- Create audit and traceability issues

Next step: Inspect duplicate rows to determine whether they are exact duplicates or conflicting records.

In [66]:
df[df["_id"].isin(["app_042", "app_001"])].sort_values("_id")

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
383,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,Female,1986-05-27,90230,...,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,NaN
455,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,NaN,NaN,NaN,NaN,NaN,...,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,DUPLICATE_ENTRY_ERROR
8,app_042,"[{'category': 'Insurance', 'amount': 153}, {'c...",NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,...,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
354,app_042,"[{'category': 'Insurance', 'amount': 153}, {'c...",NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,...,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,RESUBMISSION


## Duplicate Record Inspection

Inspection of duplicate `_id` values reveals that:

- The duplicated entries are nearly identical in structure.
- The `notes` field indicates either:
  - "DUPLICATE_ENTRY_ERROR"
  - "RESUBMISSION"

This suggests that the duplicates are not random data corruption, but rather operational artifacts.

Since `_id` should uniquely identify each application, retaining multiple rows with identical IDs would distort:
- Approval rate calculations
- Fairness metrics
- Audit traceability

Decision:
We will retain only one record per `_id`, keeping the most recent or cleanest entry.

Cleaning strategy:
Instead of dropping duplicates blindly, we will consolidate duplicates by keeping the most informative record
(the record with the highest completeness), ensuring one unique row per `_id`.

In [67]:
df["completeness_score"] = df.notna().sum(axis=1)
df.loc[df["_id"].isin(["app_042","app_001"]), ["_id","completeness_score","notes"]]

,_id,completeness_score,notes
8,app_042,15,NaN
354,app_042,16,RESUBMISSION
383,app_001,15,NaN
455,app_001,11,DUPLICATE_ENTRY_ERROR


## Deduplication Rule (Data-Driven)

To resolve duplicate `_id` values, we compute a row-level **completeness score** (number of non-null fields).

Results:
- For `app_042`, the record marked as `RESUBMISSION` has a higher completeness score (16 vs 15).
- For `app_001`, the record marked as `DUPLICATE_ENTRY_ERROR` has substantially lower completeness (11 vs 15).

Decision:
We keep **one record per `_id`**, selecting the record with the **highest completeness score**.
This preserves the most informative version of each application while ensuring unique identifiers for downstream analysis.

In [68]:
df_dedup = (
    df.sort_values(["_id", "completeness_score"], ascending=[True, False])
      .drop_duplicates(subset=["_id"], keep="first")
      .drop(columns=["completeness_score"])
)

df.shape, df_dedup.shape

((502, 22), (500, 21))

## Deduplication Result

The dataset originally contained 502 records.
After applying the completeness-based deduplication rule, the dataset now contains 500 unique application IDs.

This ensures:
- Each `_id` uniquely identifies one credit application.
- No duplicate records distort downstream approval or fairness calculations.
- The most informative record is retained in cases of operational resubmissions.

The dataset is now structurally consistent with respect to primary identifiers.

In [69]:
df_dedup["financials.annual_income"].apply(type).value_counts()

financials.annual_income
<class 'int'>      486
<class 'str'>        8
<class 'float'>      6
Name: count, dtype: int64

## Type Consistency Issue – Annual Income

The column `financials.annual_income` contains mixed data types:

- 486 integers
- 6 floats
- 8 strings

This confirms a type inconsistency issue. 
Financial variables should be strictly numeric, as mixed types may:

- Break statistical calculations
- Distort aggregation results
- Introduce silent errors in downstream fairness analysis

Plan:
Coerce the column into a unified numeric format and convert non-parsable values to null.

In [70]:
df_dedup["financials.annual_income"] = pd.to_numeric(
    df_dedup["financials.annual_income"],
    errors="coerce"
)

df_dedup["financials.annual_income"].apply(type).value_counts()

financials.annual_income
<class 'float'>    500
Name: count, dtype: int64

## Remediation – Annual Income Type Standardization

We coerced `financials.annual_income` into a numeric format using `pd.to_numeric(..., errors="coerce")`.

Result:
- The column is now consistently stored as a numeric type (float).
- Any values that could not be parsed as numbers were converted to null (NaN), preserving transparency rather than silently failing.

This ensures downstream statistical analysis and fairness metrics can be computed reliably.

In [71]:
df_dedup["financials.annual_salary"].notna().sum(), df_dedup["financials.annual_income"].isna().sum()

(np.int64(5), np.int64(5))

## Schema Consolidation – Income Variables

The dataset contains two separate income-related fields:

- `financials.annual_income`
- `financials.annual_salary`

After type coercion, we observe:

- 5 records contain `annual_salary`
- The same 5 records have missing `annual_income`

This confirms a schema inconsistency rather than independent variables.

Remediation strategy:
We will consolidate both fields into a single unified income column,
using `annual_salary` as a fallback when `annual_income` is missing.

In [72]:
df_dedup["financials.final_income"] = df_dedup["financials.annual_income"].fillna(
    df_dedup["financials.annual_salary"]
)

df_dedup["financials.final_income"].isna().sum()

np.int64(0)

## Income Consolidation Result

A unified income variable (`financials.final_income`) was created by:

- Using `annual_income` as the primary source.
- Falling back to `annual_salary` when `annual_income` was missing.

After consolidation, the final income column contains no missing values.

This resolves both:
- Type inconsistency
- Schema inconsistency

The dataset now contains a single, consistent income variable suitable for downstream modeling and fairness analysis.

In [73]:
df_clean = df_dedup.drop(columns=[
    "financials.annual_income",
    "financials.annual_salary"
])

df_clean.shape

(500, 20)

## Final Structural Cleanup

The original income-related columns have been removed to prevent ambiguity.

The dataset now contains:
- 500 unique applications
- 20 structured variables
- A single consolidated income field

This ensures schema clarity and reduces the risk of accidental misuse in downstream analysis.

In [74]:
df_clean["applicant_info.gender"].value_counts(dropna=False)

applicant_info.gender
Male      194
Female    193
F          58
M          53
            2
Name: count, dtype: int64

## Categorical Consistency Issue – Gender

The `applicant_info.gender` field contains inconsistent encodings:

- Full labels: "Male", "Female"
- Abbreviations: "M", "F"
- A small number of irregular or blank values

Such inconsistencies may distort fairness metrics if categories are treated separately.

Plan:
- Standardize values to lowercase
- Map abbreviations ("m" → "male", "f" → "female")
- Convert blank or invalid entries to null

In [75]:
df_clean["applicant_info.gender"] = (
    df_clean["applicant_info.gender"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"m": "male", "f": "female"})
)

df_clean["applicant_info.gender"].value_counts(dropna=False)

applicant_info.gender
female    251
male      247
            2
Name: count, dtype: int64

## Gender Standardization Result

The gender field has been normalized to a consistent lowercase format:

- "male"
- "female"

Abbreviations and casing inconsistencies have been resolved.

Two records remain with invalid or blank gender values. 
These will be treated as missing rather than forcefully assigned, to preserve analytical integrity.

In [76]:
df_clean["applicant_info.gender"] = df_clean["applicant_info.gender"].replace(
    ["", "nan", "none"], np.nan
)

df_clean["applicant_info.gender"].isna().sum()

np.int64(2)

## Final Gender Cleanup

Invalid or blank gender values have been converted to null (NaN).

This preserves data integrity without artificially assigning categories.

The dataset now contains:
- 251 female
- 247 male
- 2 missing

The field is now consistent and ready for fairness analysis.

In [77]:
df_clean["applicant_info.date_of_birth"].head(10)

383    1986-05-27
339    1999-08-01
284    1982-08-24
255    02/28/1995
136    1960/06/19
328    1987-07-12
12     1989-06-13
81     1993-07-12
390    1989-11-20
47     1996-09-07
Name: applicant_info.date_of_birth, dtype: object

## Date Format Inconsistency – Date of Birth

The `applicant_info.date_of_birth` field is stored as a string and contains multiple date formats:

- ISO format: `YYYY-MM-DD`
- Slash-separated format: `YYYY/MM/DD`
- US-style format: `MM/DD/YYYY`

Such inconsistencies prevent reliable parsing and can lead to incorrect age calculations.

Plan: parse the field into a proper datetime type using robust parsing rules.

In [78]:
dob_raw = (
    df_clean["applicant_info.date_of_birth"]
    .astype(str)
    .str.strip()
    .replace({"": np.nan, "nan": np.nan, "none": np.nan})
)

# Normalize separators: "/" -> "-"
dob_norm = dob_raw.str.replace("/", "-", regex=False)

# Try YYYY-MM-DD first
dob_parsed = pd.to_datetime(dob_norm, errors="coerce", format="%Y-%m-%d")

# For remaining, try DD-MM-YYYY
mask = dob_parsed.isna()
dob_parsed.loc[mask] = pd.to_datetime(dob_norm[mask], errors="coerce", format="%d-%m-%Y")

df_clean["applicant_info.date_of_birth_parsed"] = dob_parsed

df_clean["applicant_info.date_of_birth_parsed"].isna().sum()

np.int64(30)

In [79]:
mask_final = df_clean["applicant_info.date_of_birth_parsed"].isna()
mask_final.sum(), df_clean.loc[mask_final, "applicant_info.date_of_birth"].head(20)

(np.int64(30),
 255    02/28/1995
 378    08/26/1983
 441    07/15/1999
 420    03/16/1970
 26               
 275              
 451    05/21/1984
 469    11/26/1975
 348    10/24/2001
 88     12/16/1985
 462              
 313    04/13/1996
 330    09/20/1998
 498    11/15/1985
 327    06/20/2000
 345    12/13/1970
 183    06/25/1994
 226    04/25/1997
 115    12/28/1972
 134    07/28/1988
 Name: applicant_info.date_of_birth, dtype: object)

In [80]:
mask_us = dob_parsed.isna()
dob_parsed.loc[mask_us] = pd.to_datetime(dob_norm[mask_us], errors="coerce", format="%m-%d-%Y")

df_clean["applicant_info.date_of_birth_parsed"] = dob_parsed
df_clean["applicant_info.date_of_birth_parsed"].isna().sum()

np.int64(4)

In [81]:
mask_missing_dob = df_clean["applicant_info.date_of_birth_parsed"].isna()

df_clean.loc[
    mask_missing_dob,
    ["_id", "applicant_info.date_of_birth", "applicant_info.date_of_birth_parsed"]
]

,_id,applicant_info.date_of_birth,applicant_info.date_of_birth_parsed
26,app_075,,NaT
275,app_120,,NaT
462,app_165,,NaT
448,app_350,,NaT


## Date of Birth Cleaning Summary

The `applicant_info.date_of_birth` column contained multiple date formats, including:

- `YYYY-MM-DD`
- `DD/MM/YYYY`
- `MM/DD/YYYY`
- Empty strings

### Cleaning Steps

1. Empty strings were converted to `NaN`.
2. All separators (`/`) were normalized to `-`.
3. Dates were parsed using multiple format attempts:
   - `YYYY-MM-DD`
   - `DD-MM-YYYY`
   - `MM-DD-YYYY`

### Result

- 496 records were successfully parsed into proper datetime format.
- 4 records contain genuinely missing date values.
- These missing values are represented as `NaT`.

### Data Quality Impact

This process ensures:

- Consistent datetime formatting across the dataset  
- No silent parsing failures  
- Transparent handling of missing values  
- Improved reliability for downstream feature engineering  

The dataset is now ready for age feature engineering and GDPR-compliant transformation steps.

In [82]:
today = pd.Timestamp.today()

df_clean["applicant_age"] = (
    (today - df_clean["applicant_info.date_of_birth_parsed"])
    .dt.days // 365
)

df_clean["applicant_age"].describe()

count    496.000000
mean      40.747984
std       10.964912
min       23.000000
25%       32.000000
50%       39.000000
75%       47.000000
max       67.000000
Name: applicant_age, dtype: float64

## GDPR and Privacy Compliance Step

The dataset currently contains personally identifiable information (PII), including:

- Full name  
- Email address  
- Social security number (SSN)  
- IP address  
- Date of birth  

While some of these fields were necessary for data validation and feature engineering (e.g., calculating age), they are not required for downstream modeling.

To ensure GDPR compliance and privacy protection:

- All direct identifiers will be removed.
- Date of birth will be replaced by the derived feature `applicant_age`.
- System-generated notes will also be removed as they may contain sensitive operational information.

After this step, the dataset will contain only analytical and model-relevant features.

In [83]:
pii_columns = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.date_of_birth",
    "applicant_info.date_of_birth_parsed",
    "notes"
]

df_model_ready = df_clean.drop(columns=pii_columns)

df_model_ready.shape

(500, 15)

## GDPR Compliance Verification

All personally identifiable information (PII) has now been removed from the dataset.

Removed columns include:

- `applicant_info.full_name`
- `applicant_info.email`
- `applicant_info.ssn`
- `applicant_info.ip_address`
- `applicant_info.date_of_birth`
- `applicant_info.date_of_birth_parsed`
- `notes`

The dataset now contains only analytical features relevant for modeling.

Final dataset shape: **500 rows × 15 columns**

This ensures GDPR compliance and prevents exposure of sensitive personal data in downstream analytics or machine learning workflows.

In [84]:
df_model_ready.info()

<class 'pandas.core.frame.DataFrame'>
Index: 500 entries, 383 to 152
Data columns (total 15 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   _id                               500 non-null    object 
 1   spending_behavior                 500 non-null    object 
 2   processing_timestamp              62 non-null     object 
 3   applicant_info.gender             498 non-null    object 
 4   applicant_info.zip_code           500 non-null    object 
 5   financials.credit_history_months  500 non-null    int64  
 6   financials.debt_to_income         500 non-null    float64
 7   financials.savings_balance        500 non-null    int64  
 8   decision.loan_approved            500 non-null    bool   
 9   decision.rejection_reason         208 non-null    object 
 10  loan_purpose                      50 non-null     object 
 11  decision.interest_rate            292 non-null    float64
 12  decision.ap

In [85]:
df_model_ready.isna().sum().sort_values(ascending=False)

loan_purpose                        450
processing_timestamp                438
decision.rejection_reason           292
decision.interest_rate              208
decision.approved_amount            208
applicant_age                         4
applicant_info.gender                 2
_id                                   0
spending_behavior                     0
applicant_info.zip_code               0
financials.credit_history_months      0
financials.debt_to_income             0
financials.savings_balance            0
decision.loan_approved                0
financials.final_income               0
dtype: int64

## Data Quality Assessment – Completeness Analysis

Total records: **500**

### 1. loan_purpose
- Missing: 450 / 500 (90%)
- This represents extremely low completeness.
- The field may be relevant for proxy discrimination analysis.
- Indicates weak input validation or lack of mandatory field enforcement.

**Assessment:** Major data quality issue (Completeness dimension).

---

### 2. processing_timestamp
- Missing: 438 / 500 (87.6%)
- Represents a significant governance gap.

From a governance perspective:
- Weak audit trail
- Limited traceability of decision timing
- Potential non-compliance with GDPR accountability principle
- Potential transparency issues under the EU AI Act

**Assessment:** Critical governance and auditability issue.

---

### 3. decision.rejection_reason
- Missing: 292 records
- Likely corresponds to approved loans (logical absence).

This is not necessarily a completeness issue but requires a consistency check to ensure:
- Rejected loans always have a rejection reason.
- Approved loans do not incorrectly contain rejection reasons.

**Assessment:** Requires consistency validation.

---

### 4. decision.interest_rate and decision.approved_amount
- Missing: 208 records each
- Likely corresponds to rejected loans.

This is structurally expected but must be validated for logical consistency.

**Assessment:** Not inherently a data quality issue; requires validation.

---

### 5. applicant_age
- Missing: 4 (0.8%)
- Acceptable level of missingness.

---

### 6. applicant_info.gender
- Missing: 2 (0.4%)
- Low missingness but relevant for bias analysis.

---

### Summary

The most significant data quality issues relate to:

- Severe incompleteness of `loan_purpose`
- Critical governance gap in `processing_timestamp`

Other missing values appear structurally linked to decision outcomes and require consistency validation rather than remediation.

In [86]:
# Check logical consistency
pd.crosstab(
    df_model_ready["decision.loan_approved"],
    df_model_ready["decision.rejection_reason"].isna()
)

decision.rejection_reason,False,True
decision.loan_approved,,
False,208,0
True,0,292


## Logical Consistency Validation – Decision Fields

A cross-tabulation was performed between:

- `decision.loan_approved`
- Presence of `decision.rejection_reason`

### Results:

- All rejected applications (208) contain a rejection reason.
- No approved applications contain a rejection reason.
- No inconsistent cases were detected.

### Interpretation

The dataset demonstrates perfect logical consistency between loan approval status and rejection reason.

This confirms structural integrity of the decision fields and indicates no internal contradictions in outcome recording.

**Data Quality Dimension:** Consistency  
**Result:** No remediation required

In [87]:
pd.crosstab(
    df_model_ready["decision.loan_approved"],
    df_model_ready["decision.interest_rate"].isna()
)


decision.interest_rate,False,True
decision.loan_approved,,
False,0,208
True,292,0


## Logical Consistency Validation – Interest Rate

A cross-tabulation was performed between:

- `decision.loan_approved`
- Presence of `decision.interest_rate`

### Results:

- All rejected applications (208) have no interest rate.
- All approved applications (292) have a valid interest rate.
- No inconsistencies were detected.

### Interpretation

Interest rates are correctly assigned only to approved loans.

The relationship between loan approval and interest rate is logically consistent across the dataset.

**Data Quality Dimension:** Consistency  
**Result:** No remediation required

In [88]:
pd.crosstab(
    df_model_ready["decision.loan_approved"],
    df_model_ready["decision.approved_amount"].isna()
)


decision.approved_amount,False,True
decision.loan_approved,,
False,0,208
True,292,0


## Logical Consistency Validation – Approved Amount

A cross-tabulation was performed between:

- `decision.loan_approved`
- Presence of `decision.approved_amount`

### Results:

- All rejected applications (208) have no approved amount.
- All approved applications (292) contain a valid approved amount.
- No inconsistencies were detected.

### Interpretation

Approved amounts are correctly assigned only to approved loans.

The dataset demonstrates strong logical consistency between approval status and loan amount allocation.

**Data Quality Dimension:** Consistency  
**Result:** No remediation required

## Plausibility Checks (Outlier Validation)

### Debt-to-Income Ratio

In [89]:
df_model_ready["financials.debt_to_income"].describe()

count    500.000000
mean       0.245520
std        0.136148
min        0.050000
25%        0.150000
50%        0.230000
75%        0.342500
max        1.850000
Name: financials.debt_to_income, dtype: float64

In [90]:
df_model_ready[df_model_ready["financials.debt_to_income"] < 0]

,_id,spending_behavior,processing_timestamp,applicant_info.gender,applicant_info.zip_code,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.final_income,applicant_age


In [91]:
df_model_ready[df_model_ready["financials.debt_to_income"] > 1]

,_id,spending_behavior,processing_timestamp,applicant_info.gender,applicant_info.zip_code,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.final_income,applicant_age
316,app_402,"[{'category': 'Education', 'amount': 482}]",NaN,female,90214,27,1.85,37281,True,NaN,NaN,3.2,17000.0,88000.0,61.0


### Savings Balance

In [92]:
df_model_ready["financials.savings_balance"].describe()

count      500.000000
mean     29579.530000
std      16745.805308
min      -5000.000000
25%      17387.250000
50%      27399.000000
75%      38281.750000
max      88078.000000
Name: financials.savings_balance, dtype: float64

In [93]:
df_model_ready[df_model_ready["financials.savings_balance"] < 0]

,_id,spending_behavior,processing_timestamp,applicant_info.gender,applicant_info.zip_code,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.final_income,applicant_age
159,app_290,"[{'category': 'Fitness', 'amount': 599}]",NaN,female,90221,39,0.27,-5000,True,NaN,NaN,4.3,49000.0,104000.0,36.0


### Interest Rate

In [94]:
df_model_ready["decision.interest_rate"].describe()

count    292.000000
mean       4.564726
std        1.162866
min        2.500000
25%        3.500000
50%        4.550000
75%        5.600000
max        6.500000
Name: decision.interest_rate, dtype: float64

In [95]:
df_model_ready[df_model_ready["decision.interest_rate"] < 0]

,_id,spending_behavior,processing_timestamp,applicant_info.gender,applicant_info.zip_code,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.final_income,applicant_age


### Applicant Age

In [96]:
df_model_ready["applicant_age"].describe()

count    496.000000
mean      40.747984
std       10.964912
min       23.000000
25%       32.000000
50%       39.000000
75%       47.000000
max       67.000000
Name: applicant_age, dtype: float64

In [97]:
df_model_ready[
    (df_model_ready["applicant_age"] < 18) |
    (df_model_ready["applicant_age"] > 100)
]

,_id,spending_behavior,processing_timestamp,applicant_info.gender,applicant_info.zip_code,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.final_income,applicant_age


## Plausibility Check – Results

The following observations were made:

### Debt-to-Income Ratio
Values ranged from 0.05 to 1.85.
Ratios above 1 indicate high financial leverage but are economically plausible.
No invalid values were detected.

### Savings Balance
Negative balances (minimum: -5000) were observed.
These may reflect overdraft or temporary negative liquidity and are considered realistic.

### Interest Rate
Interest rates ranged between 2.5 and 6.5.
No negative or extreme values were detected.

### Applicant Age
Applicant age ranged from 23 to 67.
No unrealistic ages (<18 or >100) were found.

### Conclusion

All numerical variables fall within realistic and economically plausible ranges.
No corrective data remediation was required.